In [ ]:
import pandas as pd
import numpy as np

# =====================================================
# LOAD YOUR AUGMENTED NETWORK DATASET
# =====================================================

INPUT_FILE = "final_dataset_augmented.csv"

df = pd.read_csv(INPUT_FILE)

print("Input shape:", df.shape)

# =====================================================
# NETWORK QUALITY CALCULATOR
# =====================================================

def calculate_network_quality(row):

    score = 0

    if row["cqi"] >= 12:
        score += 2
    elif row["cqi"] >= 8:
        score += 1

    if row["snr"] >= 20:
        score += 2
    elif row["snr"] >= 10:
        score += 1

    if row["rsrp"] > -90:
        score += 2
    elif row["rsrp"] > -105:
        score += 1

    if row["dl_error"] == 0:
        score += 1

    if row["ul_error"] == 0:
        score += 1

    if score >= 6:
        return 2      # GOOD

    elif score >= 3:
        return 1      # MEDIUM

    return 0          # POOR


# =====================================================
# OFFLOADING LABEL LOGIC
# =====================================================

def decide_offloading(row):

    battery = row["battery_level"]
    cpu = row["cpu_load"]
    ram = row["ram_usage"]

    temp = row["device_temperature"]

    task = row["task_size_mb"]
    complexity = row["task_complexity"]

    latency = row["latency_requirement_ms"]

    network = row["network_quality"]

    edge_distance = row["edge_distance_km"]

    # ---------------------------------------
    # MOBILE
    # ---------------------------------------

    if (
        battery > 75 and
        cpu < 50 and
        ram < 60 and
        temp < 42 and
        task < 80 and
        complexity <= 3
    ):
        return 0

    # ---------------------------------------
    # EDGE
    # ---------------------------------------

    if (
        network >= 1 and
        latency < 50 and
        edge_distance < 15 and
        task < 500
    ):
        return 1

    # ---------------------------------------
    # CLOUD
    # ---------------------------------------

    return 2


# =====================================================
# SYNTHETIC EXPANSION
# =====================================================

np.random.seed(42)

AUGMENT_FACTOR = 5

rows = []

for _, row in df.iterrows():

    for _ in range(AUGMENT_FACTOR):

        new_row = row.copy()

        # -----------------------------------
        # DEVICE FEATURES
        # -----------------------------------

        new_row["battery_level"] = np.random.randint(5,101)

        new_row["cpu_load"] = np.random.randint(5,101)

        new_row["ram_usage"] = np.random.randint(10,101)

        new_row["device_temperature"] = round(
            np.random.uniform(30,55),2
        )

        # -----------------------------------
        # APPLICATION FEATURES
        # -----------------------------------

        new_row["task_size_mb"] = np.random.randint(1,1000)

        new_row["task_complexity"] = np.random.randint(1,11)

        new_row["latency_requirement_ms"] = np.random.randint(5,200)

        new_row["bandwidth_requirement"] = round(
            np.random.uniform(1,100),2
        )

        # -----------------------------------
        # USER CONTEXT
        # -----------------------------------

        new_row["mobility_speed"] = round(
            np.random.uniform(0,120),2
        )

        # -----------------------------------
        # EDGE/CLOUD CONTEXT
        # -----------------------------------

        new_row["edge_distance_km"] = round(
            np.random.uniform(1,30),2
        )

        new_row["cloud_distance_km"] = round(
            np.random.uniform(50,1000),2
        )

        # -----------------------------------
        # NETWORK QUALITY
        # -----------------------------------

        new_row["network_quality"] = (
            calculate_network_quality(new_row)
        )

        # -----------------------------------
        # FINAL LABEL
        # -----------------------------------

        new_row["label"] = decide_offloading(
            new_row
        )

        rows.append(new_row)

# =====================================================
# SAVE DATASET
# =====================================================

final_df = pd.DataFrame(rows)

print("\nFinal Shape:")
print(final_df.shape)

print("\nClass Distribution:")
print(final_df["label"].value_counts())

final_df.to_csv(
    "smart_offloading_dataset.csv",
    index=False
)

print("\nSaved:")
print("smart_offloading_dataset.csv")

Input shape: (94482, 11)

Final Shape:
(472410, 24)

Class Distribution:
label
2.0    451924
1.0     20147
0.0       339
Name: count, dtype: int64

Saved:
smart_offloading_dataset.csv


In [ ]:
from google.colab import files

files.download("smart_offloading_dataset.csv")

In [ ]:
import pandas as pd
import numpy as np

# =====================================================
# CONFIGURATION
# =====================================================

INPUT_FILE = "final_dataset_augmented.csv"
OUTPUT_FILE = "smart_offloading_dataset_v2.csv"

AUGMENT_FACTOR = 5

np.random.seed(42)

# =====================================================
# LOAD NETWORK DATASET
# =====================================================

df = pd.read_csv(INPUT_FILE)

print("Input Shape:", df.shape)

# =====================================================
# NETWORK QUALITY SCORE
# =====================================================

def calculate_network_quality(row):

    score = 0

    if row["cqi"] >= 12:
        score += 2
    elif row["cqi"] >= 8:
        score += 1

    if row["snr"] >= 20:
        score += 2
    elif row["snr"] >= 10:
        score += 1

    if row["rsrp"] > -90:
        score += 2
    elif row["rsrp"] > -105:
        score += 1

    if row["dl_error"] == 0:
        score += 1

    if row["ul_error"] == 0:
        score += 1

    if score >= 6:
        return 2      # GOOD

    elif score >= 3:
        return 1      # MEDIUM

    return 0          # POOR


# =====================================================
# OFFLOADING DECISION ENGINE
# =====================================================

def calculate_costs(row):

    battery = row["battery_level"]
    cpu = row["cpu_load"]
    ram = row["ram_usage"]

    task_size = row["task_size_mb"]
    complexity = row["task_complexity"]

    latency = row["latency_requirement_ms"]

    network = row["network_quality"]

    edge_distance = row["edge_distance_km"]
    cloud_distance = row["cloud_distance_km"]

    bandwidth = row["bandwidth_requirement"]

    # ----------------------------------------
    # MOBILE COST
    # ----------------------------------------

    mobile_cost = (
        task_size * 0.60 +
        complexity * 20 +
        cpu * 0.40 +
        ram * 0.10 +
        (100 - battery) * 0.70
    )

    # ----------------------------------------
    # EDGE COST
    # ----------------------------------------

    edge_cost = (
        edge_distance * 4 +
        latency * 0.50 +
        task_size * 0.15 +
        complexity * 8 -
        network * 25 -
        bandwidth * 0.20
    )

    # ----------------------------------------
    # CLOUD COST
    # ----------------------------------------

    cloud_cost = (
        cloud_distance * 0.50 +
        latency * 1.50 +
        task_size * 0.05 -
        complexity * 8 -
        bandwidth * 0.10
    )

    # ----------------------------------------
    # REALISTIC RANDOM NOISE
    # ----------------------------------------

    mobile_cost += np.random.normal(0, 15)

    edge_cost += np.random.normal(0, 15)

    cloud_cost += np.random.normal(0, 15)

    return mobile_cost, edge_cost, cloud_cost


# =====================================================
# DATASET GENERATION
# =====================================================

rows = []

generated = 0

for _, row in df.iterrows():

    for _ in range(AUGMENT_FACTOR):

        new_row = row.copy()

        # =====================================
        # DEVICE FEATURES
        # =====================================

        new_row["battery_level"] = np.random.randint(5, 101)

        new_row["cpu_load"] = np.random.randint(5, 101)

        new_row["ram_usage"] = np.random.randint(10, 101)

        new_row["device_temperature"] = round(
            np.random.uniform(30, 55), 2
        )

        # =====================================
        # APPLICATION FEATURES
        # =====================================

        new_row["task_size_mb"] = np.random.randint(1, 2000)

        new_row["task_complexity"] = np.random.randint(1, 11)

        new_row["latency_requirement_ms"] = np.random.randint(5, 200)

        new_row["bandwidth_requirement"] = round(
            np.random.uniform(1, 100), 2
        )

        # =====================================
        # USER CONTEXT
        # =====================================

        new_row["mobility_speed"] = round(
            np.random.uniform(0, 120), 2
        )

        # =====================================
        # EDGE/CLOUD CONTEXT
        # =====================================

        new_row["edge_distance_km"] = round(
            np.random.uniform(1, 30), 2
        )

        new_row["cloud_distance_km"] = round(
            np.random.uniform(50, 1000), 2
        )

        # =====================================
        # NETWORK QUALITY
        # =====================================

        new_row["network_quality"] = (
            calculate_network_quality(new_row)
        )

        # =====================================
        # COST CALCULATION
        # =====================================

        mobile_cost, edge_cost, cloud_cost = (
            calculate_costs(new_row)
        )

        new_row["mobile_cost"] = round(
            mobile_cost, 2
        )

        new_row["edge_cost"] = round(
            edge_cost, 2
        )

        new_row["cloud_cost"] = round(
            cloud_cost, 2
        )

        # =====================================
        # FINAL LABEL
        # =====================================

        costs = [
            mobile_cost,
            edge_cost,
            cloud_cost
        ]

        label = int(np.argmin(costs))

        new_row["label"] = label

        # =====================================
        # CONFIDENCE SCORE
        # =====================================

        sorted_costs = sorted(costs)

        confidence = (
            sorted_costs[1] - sorted_costs[0]
        ) / abs(sorted_costs[1])

        confidence = max(
            0,
            min(confidence, 1)
        )

        new_row["confidence"] = round(
            confidence,
            3
        )

        rows.append(new_row)

        generated += 1

        if generated % 50000 == 0:
            print("Generated:", generated)

# =====================================================
# CREATE DATAFRAME
# =====================================================

final_df = pd.DataFrame(rows)

# =====================================================
# SUMMARY
# =====================================================

print("\nFinal Shape:")
print(final_df.shape)

print("\nClass Distribution:")
print(final_df["label"].value_counts())

print("\nPercentage:")

print(
    final_df["label"]
    .value_counts(normalize=True)
    * 100
)

# =====================================================
# SAVE
# =====================================================

final_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nSaved:")
print(OUTPUT_FILE)

FileNotFoundError: [Errno 2] No such file or directory: 'final_dataset_augmented.csv'